# 02_Feature_Engineering_and_ML_Prep

Select BDA Local (Python 3.11), or the project .venv Python environment, in VS Code. Run 01, 02, and 03 sequentially. All CSV inputs, Parquet outputs, reports, and staging files use the local project root (or BDA_PROJECT_DIR). No Colab connection, uploads, exports, or Drive mounting are used. The local prototype uses four Spark threads. Missing label dates are excluded; delay defaults to timestamp.


In [1]:
import os
import sys
import json
import subprocess
import tempfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

if os.name != "nt":
    raise RuntimeError("Select the local Windows BDA Local kernel in VS Code. This notebook does not use Colab.")
# Keep all reads/writes local to this checkout; override on another device if needed.
project_hint = os.environ.get("BDA_PROJECT_DIR")
project_cwd = Path.cwd().resolve()
base_dir = (Path(project_hint).expanduser().resolve() if project_hint else
            project_cwd.parent if project_cwd.name == "notebooks" else project_cwd)
if not (base_dir / "notebooks").is_dir():
    raise FileNotFoundError("Open the project root/notebooks directory or set BDA_PROJECT_DIR to the local checkout.")
raw_dir = base_dir / "datasets_raw"
parquet_dir = base_dir / "datasets_parquet"
output_dir = base_dir / "outputs_ml_graph"
runtime_dir = base_dir / ".runtime"
temp_dir = runtime_dir / "tmp"
temp_dir.mkdir(parents=True, exist_ok=True)
for name in ["TMP", "TEMP", "TMPDIR"]:
    os.environ[name] = str(temp_dir)
tempfile.tempdir = str(temp_dir)
java_candidates = sorted(p for p in runtime_dir.glob("jdk-*") if (p / "bin/java.exe").exists())
if not java_candidates:
    raise FileNotFoundError("The project-local Java runtime is missing from .runtime/jdk-*. Complete local setup first.")
os.environ["JAVA_HOME"] = str(java_candidates[-1])
os.environ["HADOOP_HOME"] = str(runtime_dir / "hadoop")
os.environ["SPARK_HOME"] = str(runtime_dir / "spark-home")
os.environ["PATH"] = (str(java_candidates[-1] / "bin") + os.pathsep +
                      str(runtime_dir / "hadoop/bin") + os.pathsep + os.environ["PATH"])
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
required_csvs = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
missing = [name for name in required_csvs if not (raw_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing local CSVs in {raw_dir}: {missing}")
output_dir.mkdir(parents=True, exist_ok=True)
DELAY_DEFINITION = os.environ.get("BDA_DELAY_DEFINITION", "timestamp")
if DELAY_DEFINITION not in {"timestamp", "calendar_date"}:
    raise ValueError("BDA_DELAY_DEFINITION must be timestamp or calendar_date")

def ensure_package(name, required_version=None):
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = None
    if installed is None or (required_version and installed != required_version):
        spec = f"{name}=={required_version}" if required_version else name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

print(f"BDA LOCAL WINDOWS | Python: {sys.executable}")
print(f"All project data and outputs: {base_dir}")
print(f"Delay definition: {DELAY_DEFINITION}")
ensure_package("pyspark", "4.0.4")
ensure_package("pandas")
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder.master("local[4]")
         .appName("Olist_Assignment_Pipeline")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.local.dir", str(runtime_dir / "spark-local"))
         .config("spark.sql.warehouse.dir", (runtime_dir / "spark-warehouse").as_uri())
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.sql.session.timeZone", "America/Sao_Paulo")
         .getOrCreate())
print(f"Spark {spark.version}; master={spark.sparkContext.master} (single-machine prototype)")

def small_to_pandas(df):
    # Only aggregated reports are collected to the driver.
    return pd.DataFrame([row.asDict() for row in df.collect()], columns=df.columns)

def assert_unique(df, keys, name):
    if df.filter(F.expr(" OR ".join(f"`{key}` IS NULL" for key in keys))).limit(1).count():
        raise ValueError(f"{name}: null key in {keys}")
    if df.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{name}: duplicate key {keys}")

def write_feature_parquet(df):
    import shutil
    import tempfile
    destination = output_dir / "ml_delivery_delay_dataset.parquet"
    expected_rows = df.count()
    with tempfile.TemporaryDirectory(prefix="olist_features_") as work:
        staged = Path(work) / destination.name
        df.write.mode("overwrite").option("compression", "snappy").parquet(str(staged))
        if spark.read.parquet(str(staged)).count() != expected_rows:
            raise ValueError("Staged feature Parquet row count mismatch")
        if destination.resolve().parent != output_dir.resolve():
            raise ValueError("Feature output path escaped output_dir")
        if destination.is_dir():
            shutil.rmtree(destination)
        elif destination.exists():
            destination.unlink()
        shutil.copytree(staged, destination)
        parts = list(destination.glob("*.parquet"))
        if not parts or any(p.stat().st_size == 0 for p in parts):
            raise ValueError("Feature Parquet has missing/empty parts")
        if spark.read.parquet(str(destination)).count() != expected_rows:
            raise ValueError("Copied feature Parquet row count mismatch")


BDA LOCAL WINDOWS | Python: d:\Private Project\bda\BDA_Olist_Project\.venv\Scripts\python.exe
All project data and outputs: D:\Private Project\bda\BDA_Olist_Project
Delay definition: timestamp
Spark 4.0.4; master=local[4] (single-machine prototype)


In [2]:
tables = {name: spark.read.parquet(str(parquet_dir / f"olist_{name}_dataset.parquet"))
          for name in ["orders", "order_items", "order_payments", "customers", "sellers", "products"]}
df_orders, df_items, df_payments = tables["orders"], tables["order_items"], tables["order_payments"]
df_customers, df_sellers, df_products = tables["customers"], tables["sellers"], tables["products"]
for df, keys, name in [(df_orders, ["order_id"], "orders"), (df_customers, ["customer_id"], "customers"),
                       (df_sellers, ["seller_id"], "sellers"), (df_products, ["product_id"], "products"),
                       (df_items, ["order_id", "order_item_id"], "items")]:
    assert_unique(df, keys, name)

delivered_all = df_orders.filter(F.col("order_status") == "delivered")
missing_dates = delivered_all.filter(F.col("order_delivered_customer_date").isNull() |
                                     F.col("order_estimated_delivery_date").isNull()).count()
df_delivered = delivered_all.filter(F.col("order_delivered_customer_date").isNotNull() &
                                    F.col("order_estimated_delivery_date").isNotNull())

integrity = {}
for name, child, parent, key in [
    ("order_customer", df_delivered, df_customers, "customer_id"),
    ("item_order", df_items, df_orders, "order_id"),
    ("item_seller", df_items, df_sellers, "seller_id"),
    ("item_product", df_items, df_products, "product_id"),
]:
    integrity[name] = child.join(parent.select(key), on=key, how="left_anti").count()
    if integrity[name]:
        raise ValueError(f"Orphan keys found: {name}={integrity[name]}")
missing_items = df_delivered.join(df_items.select("order_id").distinct(), "order_id", "left_anti").count()
if missing_items:
    raise ValueError(f"Delivered orders without items: {missing_items}")

# One actual item row per order, deterministically chosen by minimum order_item_id.
primary_items = (df_items.withColumn("item_rank", F.row_number().over(
    Window.partitionBy("order_id").orderBy("order_item_id")))
    .filter(F.col("item_rank") == 1)
    .select("order_id", F.col("seller_id").alias("primary_seller_id"),
            F.col("product_id").alias("primary_product_id")))
items_agg = df_items.groupBy("order_id").agg(
    F.count("order_item_id").alias("total_items"), F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight"), F.countDistinct("seller_id").alias("seller_count"))
pay_agg = df_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value"),
    F.max("payment_installments").alias("max_installments"))
df_master = (df_delivered.join(items_agg, "order_id").join(primary_items, "order_id")
    .join(pay_agg, "order_id", "left").join(df_customers, "customer_id", "left")
    .join(df_sellers.select(F.col("seller_id").alias("primary_seller_id"), "seller_state"), "primary_seller_id", "left")
    .join(df_products.select(F.col("product_id").alias("primary_product_id"), "product_weight_g"), "primary_product_id", "left"))
assert_unique(df_master, ["order_id"], "feature master")
if df_master.filter(F.col("customer_state").isNull() | F.col("seller_state").isNull()).limit(1).count():
    raise ValueError("Missing seller/customer state")
timestamp_delay = F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")
calendar_delay = F.to_date("order_delivered_customer_date") > F.to_date("order_estimated_delivery_date")
delay_expression = timestamp_delay if DELAY_DEFINITION == "timestamp" else calendar_delay
imputed_weights = df_master.filter(F.col("product_weight_g").isNull()).count()
# Preserve the original 500 g fallback; this is not category-median imputation.
df_features = (df_master.withColumn("is_delayed", delay_expression.cast("int"))
    .withColumn("is_same_state", (F.col("customer_state") == F.col("seller_state")).cast("int"))
    .withColumn("freight_ratio", F.when(F.col("total_price") > 0,
                F.round(F.col("total_freight") / F.col("total_price"), 4)).otherwise(0.0))
    .withColumn("purchase_dayofweek", F.dayofweek("order_purchase_timestamp"))
    .withColumn("purchase_hour", F.hour("order_purchase_timestamp"))
    .withColumn("estimated_delivery_days", F.datediff("order_estimated_delivery_date", "order_approved_at"))
    .withColumn("product_weight_g", F.coalesce("product_weight_g", F.lit(500)))).cache()
write_feature_parquet(df_features)
state_report = small_to_pandas(df_features.groupBy("customer_state").agg(
    F.count("order_id").alias("total_orders"), F.sum("is_delayed").alias("delayed_orders"),
    F.round(F.avg("is_delayed") * 100, 2).alias("delay_rate_pct"),
    F.round(F.avg("freight_ratio"), 4).alias("avg_freight_ratio"))
    .orderBy(F.desc("delay_rate_pct"), "customer_state"))
state_report.to_csv(output_dir / "delay_by_state_report.csv", index=False)
audit = {
    "delay_definition": DELAY_DEFINITION,
    "delivered_orders_before_label_validation": delivered_all.count(),
    "excluded_missing_label_dates": missing_dates,
    "feature_orders": df_features.count(),
    "delayed_orders": df_features.filter("is_delayed = 1").count(),
    "timestamp_delayed_orders": df_master.filter(timestamp_delay).count(),
    "calendar_date_delayed_orders": df_master.filter(calendar_delay).count(),
    "weight_imputation_method": "constant_500_g", "weights_imputed": imputed_weights,
    "multi_seller_orders": df_master.filter("seller_count > 1").count(),
    "primary_item_rule": "minimum_order_item_id", "orphan_counts": integrity,
}
(output_dir / "feature_preparation_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(json.dumps(audit, indent=2))
print(state_report.head(3).to_string(index=False))


{
  "delay_definition": "timestamp",
  "delivered_orders_before_label_validation": 96478,
  "excluded_missing_label_dates": 8,
  "feature_orders": 96470,
  "delayed_orders": 7826,
  "timestamp_delayed_orders": 7826,
  "calendar_date_delayed_orders": 6534,
  "weight_imputation_method": "constant_500_g",
  "weights_imputed": 16,
  "multi_seller_orders": 1275,
  "primary_item_rule": "minimum_order_item_id",
  "orphan_counts": {
    "order_customer": 0,
    "item_order": 0,
    "item_seller": 0,
    "item_product": 0
  }
}
customer_state  total_orders  delayed_orders  delay_rate_pct  avg_freight_ratio
            AL           397              95           23.93             0.4683
            MA           717             141           19.67             0.5373
            PI           476              76           15.97             0.5029


## Class imbalance and threshold selection

Deterministic order ID hash splits give approximately 64/16/20 train/validation/test. Class weights use training labels only. Candidate and threshold are selected by maximum validation F1, then frozen before test evaluation. Compare with an unweighted model on the same test orders. Split membership is saved for reproducibility. Previous randomSplit membership was not saved, so historical metrics are not paired comparisons. Scores are not calibrated probabilities; F1 assumes no specified business cost.

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
import numpy as np
from scipy.integrate import trapezoid

# Prediction after approval. Only training labels determine class weights.
# Candidate and threshold selection uses validation labels, never test labels.
feature_cols = ["total_items", "total_price", "total_freight", "freight_ratio", "max_installments",
                "estimated_delivery_days", "is_same_state", "purchase_dayofweek", "purchase_hour", "product_weight_g"]
df_ml = df_features.select(["order_id"] + feature_cols + ["is_delayed"]).dropna().orderBy("order_id").cache()
eligible_count = df_ml.count()
excluded_count = df_features.count() - eligible_count
outer_bucket = F.pmod(F.xxhash64(F.col("order_id"), F.lit(42)), F.lit(10000))
development_df = df_ml.filter(outer_bucket < 8000)
test_df = df_ml.filter(outer_bucket >= 8000)
development_df = development_df.cache()
development_count = development_df.count()
# Stable inner assignment by order ID avoids repartition-dependent random sampling.
validation_bucket = F.pmod(F.xxhash64(F.col("order_id"), F.lit(43)), F.lit(10000))
train_df = development_df.filter(validation_bucket < 8000)
validation_df = development_df.filter(validation_bucket >= 8000)
train_df, validation_df, test_df = train_df.cache(), validation_df.cache(), test_df.cache()
train_count, validation_count, test_count = train_df.count(), validation_df.count(), test_df.count()
for name, split in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    if split.select("is_delayed").distinct().count() != 2:
        raise ValueError(f"Both classes must occur in {name}")
    if split.select("order_id").distinct().count() != split.count():
        raise ValueError(f"Duplicate orders in {name}")
print(f"Split counts: eligible={eligible_count}, development={development_count}, train={train_count}, validation={validation_count}, test={test_count}", flush=True)
assert train_count + validation_count == development_count
assert development_count + test_count == eligible_count
assert train_df.join(validation_df, "order_id", "inner").limit(1).count() == 0
assert development_df.join(test_df, "order_id", "inner").limit(1).count() == 0
split_manifest = train_df.select("order_id").withColumn("split", F.lit("train")).unionByName(
    validation_df.select("order_id").withColumn("split", F.lit("validation"))).unionByName(
    test_df.select("order_id").withColumn("split", F.lit("test")))
small_to_pandas(split_manifest).sort_values("order_id").to_csv(
    output_dir / "model_data_split_manifest.csv", index=False)
train_positive = train_df.filter("is_delayed = 1").count()
weights = {0: train_count / (2 * (train_count - train_positive)),
           1: train_count / (2 * train_positive)}

def scored(model, frame):
    return model.transform(frame).select("order_id", "is_delayed",
        vector_to_array("probability")[1].alias("delay_score"))

def threshold_table(scores):
    # Exact sweep over distinct validation scores; tied scores remain together.
    data = scores.sort_values("delay_score", ascending=False, kind="stable")
    y = data.is_delayed.to_numpy(dtype=int)
    p = data.delay_score.to_numpy(dtype=float)
    assert len(y) and np.isfinite(p).all() and set(y) == {0, 1}
    end = np.r_[np.flatnonzero(p[:-1] != p[1:]), len(p) - 1]
    tp = np.cumsum(y)[end]
    fp = end + 1 - tp
    fn = y.sum() - tp
    precision = tp / (tp + fp)
    recall = tp / y.sum()
    f1 = 2 * tp / (2 * tp + fp + fn)
    return pd.DataFrame({"threshold": p[end], "precision": precision, "recall": recall,
                         "f1": f1, "tp": tp, "fp": fp, "fn": fn})

def classification_metrics(data, threshold):
    y = data.is_delayed.to_numpy(dtype=int)
    pred = data.delay_score.to_numpy() >= threshold
    tn = int(((y == 0) & ~pred).sum()); fp = int(((y == 0) & pred).sum())
    fn = int(((y == 1) & ~pred).sum()); tp = int(((y == 1) & pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"threshold": float(threshold), "accuracy": (tn + tp) / len(y),
            "delay_precision": precision, "delay_recall": recall,
            "delay_f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
            "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}}

models = {}
comparison = []
curves = []
for candidate in ["unweighted", "balanced_class_weights"]:
    print(f"Fitting {candidate} on training data; selecting threshold on validation data", flush=True)
    training = train_df.withColumn("class_weight", F.when(F.col("is_delayed") == 1,
        F.lit(weights[1] if candidate == "balanced_class_weights" else 1.0)).otherwise(
        F.lit(weights[0] if candidate == "balanced_class_weights" else 1.0)))
    pipeline = Pipeline(stages=[VectorAssembler(inputCols=feature_cols, outputCol="features"),
        RandomForestClassifier(labelCol="is_delayed", featuresCol="features", weightCol="class_weight",
                               numTrees=100, maxDepth=8, seed=42)])
    model = pipeline.fit(training)
    models[candidate] = model
    val_scores = small_to_pandas(scored(model, validation_df))
    curve = threshold_table(val_scores)
    best = curve.sort_values(["f1", "precision", "threshold"], ascending=False).iloc[0]
    default = classification_metrics(val_scores, 0.5)
    comparison.append({"candidate": candidate, "selected_threshold": float(best.threshold),
        "validation_f1": float(best.f1), "validation_precision": float(best.precision),
        "validation_recall": float(best.recall), "validation_default_f1": default["delay_f1"],
        "validation_default_recall": default["delay_recall"]})
    curve.insert(0, "candidate", candidate)
    curves.append(curve)
comparison_df = pd.DataFrame(comparison).sort_values(
    ["validation_f1", "validation_precision"], ascending=False, kind="stable")
selected = comparison_df.iloc[0]
selected_name = str(selected.candidate)
selected_threshold = float(selected.selected_threshold)
model = models[selected_name]
comparison_df.to_csv(output_dir / "model_validation_comparison.csv", index=False)
pd.concat(curves, ignore_index=True).to_csv(output_dir / "model_validation_thresholds.csv", index=False)

# Freeze candidate and threshold before inspecting test outcomes. No test-driven retuning.
test_scores = small_to_pandas(scored(model, test_df))
tuned = classification_metrics(test_scores, selected_threshold)
default = classification_metrics(test_scores, 0.5)
unweighted_default = classification_metrics(small_to_pandas(scored(models["unweighted"], test_df)), 0.5)
# Exact ranking metrics with ties handled together; PR area uses trapezoidal interpolation
# as Spark areaUnderPR does, and is distinct from average precision.
ranked = test_scores.sort_values("delay_score", ascending=False, kind="stable")
y = ranked.is_delayed.to_numpy(dtype=int); p = ranked.delay_score.to_numpy()
ends = np.r_[np.flatnonzero(p[:-1] != p[1:]), len(p) - 1]
tp = np.cumsum(y)[ends]; fp = ends + 1 - tp
tpr = np.r_[0.0, tp / y.sum()]; fpr = np.r_[0.0, fp / (len(y) - y.sum())]
precision_curve = tp / (tp + fp)
roc_auc = float(trapezoid(tpr, fpr))
pr_auc = float(trapezoid(np.r_[precision_curve[0], precision_curve], np.r_[0.0, tp / y.sum()]))
majority_label = int(train_positive > train_count / 2)
baseline_accuracy = float((test_scores.is_delayed == majority_label).mean())
metrics = {"spark_version": spark.version, "seed": 42, "delay_definition": DELAY_DEFINITION,
    "feature_columns": feature_cols, "num_trees": 100, "max_depth": 8,
    "eligible_orders": eligible_count, "excluded_missing_predictors": excluded_count,
    "train_orders": train_count, "validation_orders": validation_count, "test_orders": test_count,
    "selected_candidate": selected_name, "selected_threshold": selected_threshold,
    "selection_objective": "maximum validation F1; ties prefer precision then higher threshold",
    "class_weights_from_train_only": {str(k): v for k, v in weights.items()},
    "roc_auc": roc_auc, "pr_auc": pr_auc, "positive_test_prevalence": float(y.mean()),
    **tuned, "default_threshold_test_metrics": default,
    "unweighted_default_threshold_test_metrics": unweighted_default,
    "majority_baseline_label": majority_label, "majority_baseline_accuracy": baseline_accuracy,
    "majority_baseline_delay_f1": 0.0 if majority_label == 0 else 2 * y.mean() / (1 + y.mean()),
    "split_method": "deterministic order_id xxhash64 salt 42 outer 80/20 and salt 43 inner 80/20; approximately 64/16/20 overall",
    "previous_run_comparability": "Previous randomSplit holdout membership was not saved and may vary; use the unweighted comparator on this same test partition",
    "limitations": "F1 assumes no explicit business cost; scores are not calibrated; future-period generalization not tested"}
(output_dir / "model_evaluation_report.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
pred = (test_scores.delay_score >= selected_threshold).astype(int)
pd.DataFrame({"is_delayed": test_scores.is_delayed, "prediction": pred}).groupby(
    ["is_delayed", "prediction"]).size().reset_index(name="count").to_csv(
    output_dir / "model_confusion_matrix.csv", index=False)
importance = pd.DataFrame({"Feature": feature_cols, "Importance": model.stages[-1].featureImportances.toArray()})
importance.sort_values("Importance", ascending=False).to_csv(output_dir / "feature_importance_report.csv", index=False)
print(comparison_df.to_string(index=False))
print(json.dumps(metrics, indent=2))
for frame in [train_df, validation_df, test_df, development_df, df_ml, df_features]:
    frame.unpersist()


Split counts: eligible=96455, development=77339, train=61826, validation=15513, test=19116
Fitting unweighted on training data; selecting threshold on validation data
Fitting balanced_class_weights on training data; selecting threshold on validation data
             candidate  selected_threshold  validation_f1  validation_precision  validation_recall  validation_default_f1  validation_default_recall
            unweighted            0.099536       0.216822              0.142929           0.448899               0.000000                   0.000000
balanced_class_weights            0.558782       0.214393              0.140981           0.447327               0.202013                   0.631289
{
  "spark_version": "4.0.4",
  "seed": 42,
  "delay_definition": "timestamp",
  "feature_columns": [
    "total_items",
    "total_price",
    "total_freight",
    "freight_ratio",
    "max_installments",
    "estimated_delivery_days",
    "is_same_state",
    "purchase_dayofweek",
    "purchase_